<a href="https://colab.research.google.com/github/aounraza379/research_paper_intelligence_assistant/blob/main/research_paper_intelligence_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research Paper Intelligence Assistant - System 1 (Foundation RAG)

A RAG (Retrieval-Augmented Generation) pipeline built from scratch - no LangChain -
to understand each component before using framework shortcuts.

**Pipeline:** PDF ingestion → chunking → embeddings → vector search (FAISS) → LLM generation → citation tracking

See `README.md` in this repo for the full architecture diagram and known limitations.


## Step 1 - Environment Setup
Install Groq client and connect using an API key stored in Colab Secrets (never hardcoded)

In [ ]:
!pip install groq -q

In [ ]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

In [ ]:
# Testing API work
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in 6 words."}]
)

print(response.choices[0].message.content)

## Step 2 - PDF Ingestion & Text Extraction
Convert a PDF into plain text. Extraction quality is inspected per-page, since messy extraction (tables, multi-column layouts) propagates into every later step

In [ ]:
!pip install pymupdf -q # handles text extraction more reliably and is faster

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload PDF

In [ ]:
#  Text Extracter
import fitz  # PyMuPDF

filename = list(uploaded.keys())[0]  # uploaded file
print(filename) # our file name
doc = fitz.open(filename)

full_text = ""
for page_num, page in enumerate(doc):
    page_text = page.get_text()
    full_text += page_text
    print(f"--- Page {page_num + 1} preview ---")
    print(page_text[:500])  # first 500 chars of each page for inspection
    print()

print(f"\nTotal characters extracted: {len(full_text)}")

## Step 3 - Document Chunking
Split the extracted text into overlapping chunks. Character-based splitting (not sentence-aware) - a known, accepted limitation for System 1, documented in the README

In [ ]:
# Document Chunking
def chunk_text(text, chunk_size=1000, overlap=200):
    """
    Splits text into overlapping chunks.
    chunk_size: max characters per chunk
    overlap: characters repeated between consecutive chunks
    """
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap  # overlapping

    return chunks

chunks = chunk_text(full_text, chunk_size=1000, overlap=200)

print(f"Total chunks created: {len(chunks)}")
print("\n--- First chunk ---")
print(chunks[0])
print("\n--- Second chunk (notice overlap with end of first) ---")
print(chunks[1])

## Step 4 - Embeddings
Convert each text chunk into a numeric vector that captures meaning, using a local Sentence-Transformers model (no extra API key needed)

In [ ]:
# Install Sentence Transformers
!pip install sentence-transformers -q

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed just the first chunk to see what an embedding actually looks like
sample_embedding = model.encode(chunks[0])

print(f"Embedding shape: {sample_embedding.shape}")
print(f"First 10 values: {sample_embedding[:10]}")

### Learning / demo only - not part of the production pipeline
The next two cells build intuition for *what* an embedding is and *why* semantic similarity works, before embedding the full document. Safe to skip on re-run.

In [ ]:
# DEMO: inspect a single embedding
sample_embedding = model.encode(chunks[0])
print(f"Embedding shape: {sample_embedding.shape}")
print(f"First 10 values: {sample_embedding[:10]}")

In [ ]:
# Testing semantic similarity works!
from sklearn.metrics.pairwise import cosine_similarity

test_sentences = [
    "Sybil attacks use fake identities to manipulate systems.",
    "Adversaries deploy multiple fabricated accounts to inject misinformation.",  # similar meaning, different words
    "The weather in Karachi is hot in summer."  # unrelated
]

test_embeddings = model.encode(test_sentences)

sim_1_2 = cosine_similarity([test_embeddings[0]], [test_embeddings[1]])[0][0]
sim_1_3 = cosine_similarity([test_embeddings[0]], [test_embeddings[2]])[0][0]

print(f"Similarity (Sybil sentence vs. paraphrased Sybil sentence): {sim_1_2:.3f}")
print(f"Similarity (Sybil sentence vs. unrelated weather sentence): {sim_1_3:.3f}")

In [ ]:
# Embed all chunks - actual pipeline step
chunk_embeddings = model.encode(chunks, show_progress_bar=True)

print(f"Total embeddings created: {len(chunk_embeddings)}")
print(f"Shape of embeddings array: {chunk_embeddings.shape}")

## Step 5 - Vector Database (FAISS) + Semantic Retrieval
Index all chunk embeddings for fast nearest-neighbor search, and write a function to retrieve the most relevant chunks for a query.

In [ ]:
# Install FAISS
!pip install faiss-cpu -q

In [ ]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)   # L2 = Euclidean distance between vectors
index.add(np.array(chunk_embeddings))

print(f"Total vectors in index: {index.ntotal}")

In [ ]:
# Retrieval function
def retrieve(query, top_k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "rank": rank + 1,
            "chunk": chunks[idx],
            "distance": distances[0][rank]
        })
    return results

In [ ]:
# Test with a real question about paper
query = "How does the system detect GPS-spoofing attackers?"
results = retrieve(query, top_k=5)

for r in results:
    print(f"Rank {r['rank']} (distance: {r['distance']:.3f})")
    print(r['chunk'][:300])
    print("---")

## Step 6 - LLM Response Generation
Feed retrieved chunks to the LLM as grounding context. Low temperature + explicit "say so if you don't know" instruction reduces hallucination risk

In [ ]:
# generation function
def generate_answer(query, top_k=3):
    retrieved = retrieve(query, top_k=top_k)

    context = "\n\n".join([f"[Chunk {r['rank']}]: {r['chunk']}" for r in retrieved])

    prompt = f"""Answer the question using ONLY the context below.
If the context doesn't contain enough information to answer, say so explicitly — do not guess or use outside knowledge.

Context:
{context}

Question: {query}

Answer:"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1  # low temperature = more grounded, less creative/hallucinatory
    )

    return response.choices[0].message.content, retrieved

In [ ]:
# Test it on the same GPS-spoofing question
answer, sources = generate_answer("How does the system detect GPS-spoofing attackers?")
print("ANSWER:")
print(answer)

In [ ]:
# Test it on something the paper likely does NOT cover - to check hallucination behavior
answer2, sources2 = generate_answer("What programming language was used to build the mobile app UI?")
print("ANSWER:")
print(answer2)

## Step 7 - Citation / Source Tracking
Two layers of citation are tracked separately:
- **LLM self-reported sources** - the model's own claim of what it used (can be wrong / over-cited).
- **Ground-truth retrieval trail** - the actual chunks mathematically retrieved and fed to the model (reliable, independently verifiable).

Testing showed these can disagree (documented in README under Known Limitations)

In [ ]:
# Updated function to return structured citations
def generate_answer_with_citations(query, top_k=3):
    retrieved = retrieve(query, top_k=top_k)

    context = "\n\n".join([f"[Chunk {r['rank']}]: {r['chunk']}" for r in retrieved])

    prompt = f"""Answer the question using ONLY the context below.
If the context doesn't contain enough information to answer, say so explicitly.
After your answer, on a new line, state which chunk number(s) you used, like: "Sources: [Chunk 1, Chunk 3]"

Context:
{context}

Question: {query}

Answer:"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )

    answer_text = response.choices[0].message.content

    return {
        "answer": answer_text,
        "retrieved_chunks": retrieved  # full chunk text + distance, for verification
    }

In [ ]:
# Test it and inspect the full source trail
result = generate_answer_with_citations("How does the system detect GPS-spoofing attackers?")

print("ANSWER:")
print(result["answer"])

print("\n--- RETRIEVED SOURCE CHUNKS (for verification) ---")
for r in result["retrieved_chunks"]:
    print(f"[Chunk {r['rank']}] (distance: {r['distance']:.3f})")
    print(r["chunk"][:200])
    print()

## Step 8 - End-to-End Pipeline + Stress Test
Wrap everything into one reusable function, then test across varied question types: factual, numeric, conceptual, out-of-scope, and broad/ambiguous.

In [ ]:
def rag_pipeline(query, top_k=3, verbose=True):
    result = generate_answer_with_citations(query, top_k=top_k)

    if verbose:
        print(f"QUESTION: {query}\n")
        print(f"ANSWER:\n{result['answer']}\n")
        print("VERIFIED SOURCE CHUNKS:")
        for r in result['retrieved_chunks']:
            print(f"  [Chunk {r['rank']}] distance={r['distance']:.3f} — {r['chunk'][:100]}...")
        print("\n" + "="*80 + "\n")

    return result

In [ ]:
# Stress-test with varied question types
rag_pipeline("What is the Haversine formula used for in this paper?")

# Numeric/results question
rag_pipeline("What percentage of Profile α Sybil posts were detected?")

# Conceptual/comparative question
rag_pipeline("Why don't traditional reputation systems work in disaster response?")

# Out-of-scope question (hallucination check, again)
rag_pipeline("Who are the authors of this paper and their university affiliations?")

# Ambiguous/broad question
rag_pipeline("Is this system effective?")

## Step 9 - Wrap-Up

**Diagnosis check:** the "authors/affiliations" question above returned "not in context" - but this is
a **retrieval miss**, not a genuine out-of-scope case. The cell below confirms whether author/affiliation
text actually exists in the early chunks (it's usually short, entity-dense text that embeds poorly against
natural-language queries - see README, Known Limitations)

In [ ]:
# Diagnostic: check if author/affiliation info exists in the source text but failed to retrieve
for i, c in enumerate(chunks[:2]):
    print(f"--- Chunk {i} ---")
    print(c[:400])
    print()

In [ ]:
# Persist the index + chunks so the notebook doesn't need to re-embed from scratch on reload
import pickle

faiss.write_index(index, "spatial_trust.index")

with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Saved: spatial_trust.index, chunks.pkl")

**To reload in a fresh session** (skip re-running Steps 2-5):
```python
index = faiss.read_index("spatial_trust.index")
with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)
```